<a href="https://colab.research.google.com/github/yoshiagent/mathModel-colab/blob/main/notebooks/O1_FirstOrder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 一階系統 First Order System

對照網頁：[數學建模｜一階系統](https://yoshiagent.github.io/mathModel/O1.html)

原課程以 MATLAB／Simulink 示範一階系統 $T\dot y + y = f(t)$ 的步階響應、自然響應、轉移函數、Bode 圖與濾波。
這本筆記本把原課程的程式逐一改寫成 Python：

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `order1.m` | $y = 1-e^{-t/T}$，T = 1、2、3 | 第 1 節 |
| `Order1Sybolic.m` | 符號解 `dsolve` | 第 1 節（sympy） |
| `order1a.m` | 自然響應 $y = e^{-t/T}$ | 第 2 節 |
| `ltiview`、`step(sys1)` | 上升時間、安定時間 | 第 3 節 |
| `bode(sys1)`、`bandwidth` | Bode 圖與頻寬 | 第 4 節 |
| `order1Bode_01.mdl`、`order1FR_01.m`、`FreqResp_01.m` | 正弦輸入的穩態響應 | 第 5 節 |
| `order1b.m` | $3/(2s+1)$：DC gain 與終值定理 | 第 6 節 |
| `order1c.m` | $s/(2s+1)$：高通濾波 | 第 7 節 |
| `order1Filter.mdl` | 低通濾波（0.1 與 20 rad/s 兩個正弦波相加） | 第 8 節 |
| Bode 漸近線、積分器 | 漸近線與實際曲線、二階系統、範例 | 第 9 節 |

只用 Colab 內建的 numpy、scipy、sympy、plotly。圖上的文字用英文（Colab 預設字型沒有中文）。

In [1]:
import numpy as np
import sympy as sp
from scipy import signal
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab, ylab, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def ds(x, n=600):
    """畫圖用的降取樣（避免筆記本檔案過大）"""
    idx = np.linspace(0, len(x) - 1, min(n, len(x))).astype(int)
    return np.round(np.asarray(x)[idx], 5)

## 1. 步階響應 $T\dot y + y = u_s(t),\ y(0)=0$

解析解：$y_h = c\,e^{-t/T}$、$y_p = 1$，由 $y(0)=0$ 得 $c=-1$，所以 $y = 1 - e^{-t/T}$。

原 `order1.m`：
```matlab
t=0:0.01:10;
y1=1-exp(-t);
y2=1-exp(-1/2*t);
y3=1-exp(-1/3*t);
plot(t,y1,'r',t,y2,'g',t,y3,'b')
grid
```

In [2]:
t = np.arange(0, 10.001, 0.01)
fig = go.Figure()
for T, c in [(1, "red"), (2, "green"), (3, "blue")]:
    fig.add_scatter(x=ds(t), y=ds(1 - np.exp(-t / T)), name=f"T = {T}", line=dict(color=c))
    fig.add_scatter(x=[T], y=[1 - np.exp(-1)], mode="markers", marker=dict(color="black", size=8), showlegend=False)
show(fig, "Step response y = 1 - exp(-t/T)  (y(T) = 0.6321)", "t (s)", "y")

原 `Order1Sybolic.m` 用 Symbolic Math Toolbox 的 `dsolve` 求解；Python 用 sympy：
```matlab
y=dsolve('T*Dy+y=1','y(0)=0','t')
```

In [3]:
tt, T = sp.symbols("t T", positive=True)
y = sp.Function("y")
sol = sp.dsolve(sp.Eq(T * y(tt).diff(tt) + y(tt), 1), y(tt), ics={y(0): 0})
print(sol)                                    # y(t) = 1 - exp(-t/T)
ysym = sol.rhs
for k in (1, 3, 4, 5):                        # 1T、3T、4T、5T 時的數值（網頁上的表）
    print(f"y({k}T) = 1 - e^-{k} = {float(ysym.subs(tt, k * T)):.4f}")
t09 = float(sp.solve(sp.Eq(ysym, sp.Rational(9, 10)), tt)[0].subs(T, 1))
t01 = float(sp.solve(sp.Eq(ysym, sp.Rational(1, 10)), tt)[0].subs(T, 1))
print(f"t0.9 = {t09:.4f} T, t0.1 = {t01:.4f} T, rise time tr = {t09 - t01:.4f} T")

Eq(y(t), 1 - exp(-t/T))
y(1T) = 1 - e^-1 = 0.6321
y(3T) = 1 - e^-3 = 0.9502
y(4T) = 1 - e^-4 = 0.9817
y(5T) = 1 - e^-5 = 0.9933
t0.9 = 2.3026 T, t0.1 = 0.1054 T, rise time tr = 2.1972 T


## 2. 自然響應 $T\dot y + y = 0,\ y(0) = 1$ → $y = e^{-t/T}$（原 `order1a.m`）

In [4]:
fig = go.Figure()
for T_, c in [(1, "red"), (2, "green"), (3, "blue")]:
    fig.add_scatter(x=ds(t), y=ds(np.exp(-t / T_)), name=f"T = {T_}", line=dict(color=c))
    fig.add_scatter(x=[T_], y=[np.exp(-1)], mode="markers", marker=dict(color="black", size=8), showlegend=False)
show(fig, "Natural response y = exp(-t/T)  (y(T) = 0.3679)", "t (s)", "y")
print(sp.dsolve(sp.Eq(T * y(tt).diff(tt) + y(tt), 0), y(tt), ics={y(0): 1}))

Eq(y(t), exp(-t/T))


## 3. 轉移函數與步階響應特性

$$H(s) = \frac{Y(s)}{F(s)} = \frac{1}{Ts+1},\qquad \text{pole } s = -\frac{1}{T}$$

原講義在 MATLAB 輸入 `s=tf('s'); sys1=1/(2*s+1); step(sys1)`，在圖上按右鍵勾選 Rise Time 與 Settling Time，得到 4.39 s 與 7.82 s。
MATLAB 的上升時間是 10%→90%、安定時間以 2% 為準。

In [5]:
def step_info(num, den, tf=40, n=40001):
    tt_, yy = signal.step(signal.lti(num, den), T=np.linspace(0, tf, n))
    yf = yy[-1]
    tr = tt_[np.argmax(yy >= 0.9 * yf)] - tt_[np.argmax(yy >= 0.1 * yf)]
    out = np.where(np.abs(yy - yf) > 0.02 * abs(yf))[0]
    ts = tt_[out[-1] + 1]
    return tt_, yy, tr, ts


ts_, ys_, tr, tset = step_info([1], [2, 1])
print(f"sys1 = 1/(2s+1): rise time = {tr:.2f} s (2.2T = {2 * np.log(9):.2f}), settling time (2%) = {tset:.3f} s (T ln50 = {2 * np.log(50):.3f})")
fig = go.Figure()
k = ts_ <= 12
fig.add_scatter(x=ds(ts_[k]), y=ds(ys_[k]), name="1/(2s+1)")
fig.add_scatter(x=[2], y=[1 - np.exp(-1)], mode="markers+text", text=["t=T=2, y=0.632"], textposition="bottom right", showlegend=False)
show(fig, "Step response of 1/(2s+1)", "t (s)", "Amplitude")

sys1 = 1/(2s+1): rise time = 4.39 s (2.2T = 4.39), settling time (2%) = 7.825 s (T ln50 = 7.824)


`ltiview` 比較 $T$ = 1、2、3 三個系統（原講義：`sys1=tf(1,[1 1]); sys2=tf(1,[2 1]); sys3=tf(1,[3 1]); ltiview`）：

In [6]:
fig = go.Figure()
for T_ in (1, 2, 3):
    tt_, yy, tr_, ts2 = step_info([1], [T_, 1])
    k = tt_ <= 15
    fig.add_scatter(x=ds(tt_[k]), y=ds(yy[k]), name=f"1/({T_}s+1): tr={tr_:.2f}, ts={ts2:.2f}")
show(fig, "ltiview: step responses for T = 1, 2, 3", "t (s)", "Amplitude")

## 4. Bode 圖與頻寬

$$|H(j\omega)| = \left(1+\omega^2T^2\right)^{-1/2},\qquad 20\log|H(j\omega)| = -10\log\left(1+\omega^2T^2\right)$$

轉角頻率 $\omega = 1/T$ 時下降 3 dB、相位 −45°。原講義：`bode(sys1)`，`bandwidth(sys1)` → 0.4988。

In [7]:
w = np.logspace(-2, 1, 400)
_, mag, ph = signal.bode(signal.lti([1], [2, 1]), w)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_scatter(x=w, y=np.round(mag, 4), name="Magnitude (dB)", row=1, col=1)
fig.add_scatter(x=w, y=np.round(ph, 4), name="Phase (deg)", row=2, col=1)
fig.add_scatter(x=[0.5], y=[-3.01], mode="markers+text", text=["w=0.5, -3.01 dB"], textposition="top right", showlegend=False, row=1, col=1)
fig.add_scatter(x=[0.5], y=[-45], mode="markers+text", text=["-45 deg"], textposition="top right", showlegend=False, row=2, col=1)
fig.update_xaxes(type="log", title="Frequency (rad/s)", row=2, col=1)
fig.update_yaxes(title="dB", row=1, col=1)
fig.update_yaxes(title="deg", row=2, col=1)
show(fig, "Bode diagram of 1/(2s+1)", None, None, h=560)
bw = np.sqrt(10 ** 0.3 - 1) / 2                   # |H| 降到 10^(-3/20) 的頻率
print(f"bandwidth = {bw:.4f} rad/s   (MATLAB bandwidth(sys1) -> 0.4988; 1/T = 0.5)")
print(f"10^(-3/20) = {10 ** (-3 / 20):.4f}")

bandwidth = 0.4988 rad/s   (MATLAB bandwidth(sys1) -> 0.4988; 1/T = 0.5)
10^(-3/20) = 0.7079


## 5. 正弦輸入的穩態響應（`order1Bode_01.mdl`、`order1FR_01.m`）

Simulink 模型：Sine Wave（頻率 0.5 rad/s）→ Transfer Fcn $\frac{1}{2s+1}$，模擬 100 s（ode45、最大步長 0.005），
用 To Workspace 存 t、u、y，再以 `plot(t,u,'r',t,y,'b')` 畫圖。
原講義在圖上量到：峰值時間 65.97 → 78.55（週期 $4\pi$）、輸出落後 1.58 s、振幅 0.7071。

In [8]:
t5 = np.arange(0, 100.0001, 0.005)
u5 = np.sin(0.5 * t5)
_, y5, _ = signal.lsim(signal.lti([1], [2, 1]), u5, t5)
k = t5 > 60                                         # 暫態消失後
amp = y5[k].max()
pk_u = t5[k][np.argmax(u5[k])]
pk_y = t5[(t5 > pk_u) & (t5 < pk_u + 6)][np.argmax(y5[(t5 > pk_u) & (t5 < pk_u + 6)])]
print(f"output amplitude = {amp:.4f} (1/sqrt(2) = {1 / np.sqrt(2):.4f})")
print(f"input peak t = {pk_u:.2f}, output peak t = {pk_y:.2f}, lag = {pk_y - pk_u:.3f} s, phase = {0.5 * (pk_y - pk_u) * 180 / np.pi:.1f} deg")
fig = go.Figure()
fig.add_scatter(x=ds(t5, 1500), y=ds(u5, 1500), name="u = sin(0.5t)", line=dict(color="red"))
fig.add_scatter(x=ds(t5, 1500), y=ds(y5, 1500), name="y", line=dict(color="blue"))
show(fig, "Sine input through 1/(2s+1): steady-state gain 0.707, phase lag 45 deg", "t (s)", "")

output amplitude = 0.7071 (1/sqrt(2) = 0.7071)
input peak t = 78.54, output peak t = 80.11, lag = 1.570 s, phase = 45.0 deg


原 `FreqResp_01.m` 示範相位差：`y2=sin(2*t-pi/2)` 比 `y1=sin(2*t)` 晚 $\pi/4$ 秒，相位差 $2\times\pi/4 = \pi/2$。

In [9]:
t6 = np.arange(0, 10.001, 0.01)
fig = go.Figure()
fig.add_scatter(x=t6, y=np.round(np.sin(2 * t6), 4), name="sin(2t)", line=dict(color="red"))
fig.add_scatter(x=t6, y=np.round(np.sin(2 * t6 - np.pi / 2), 4), name="sin(2t - pi/2)", line=dict(color="blue"))
show(fig, "FreqResp_01.m: time shift pi/4 s = phase pi/2 at w = 2", "t (s)", "")

## 6. DC gain 與終值定理（`order1b.m`：$sys1 = \frac{3}{2s+1}$）

$$\lim_{t\to\infty}y(t) = \lim_{s\to0} sY(s) = \lim_{s\to 0} s\cdot\frac{1}{s}\cdot\frac{3}{2s+1} = 3$$

In [10]:
s = sp.symbols("s")
print("final value =", sp.limit(s * (1 / s) * 3 / (2 * s + 1), s, 0))
print(f"DC gain in dB = 20 log10(3) = {20 * np.log10(3):.4f}")
t7, y7 = signal.step(signal.lti([3], [2, 1]), T=np.linspace(0, 12, 1201))
print(f"y(T=2) = {np.interp(2, t7, y7):.4f}  (3 x 0.632 = 1.896)")
fig = go.Figure()
fig.add_scatter(x=ds(t7), y=ds(y7), name="3/(2s+1)")
show(fig, "Step response of 3/(2s+1): final value 3, y(2) = 1.896", "t (s)", "Amplitude")

final value = 3
DC gain in dB = 20 log10(3) = 9.5424
y(T=2) = 1.8964  (3 x 0.632 = 1.896)


## 7. 高通濾波（`order1c.m`：$sys2 = \frac{s}{2s+1}$）

原檔定義了 `sys2`，但後面 `bode(sys1)`、`step(sys1)` 仍用 `sys1`；因為開頭有 `clear all`，在 MATLAB 會出現「未定義 sys1」的錯誤。
這裡改用 `sys2`。

初值定理 $\lim_{t\to0}y = \lim_{s\to\infty} s\cdot\frac{1}{s}\cdot\frac{s}{2s+1} = 0.5$；終值 0；高頻增益 $20\log(0.5) = -6.02$ dB。

In [11]:
print("initial value =", sp.limit(s * (1 / s) * s / (2 * s + 1), s, sp.oo), " final value =", sp.limit(s * (1 / s) * s / (2 * s + 1), s, 0))
_, mag2, ph2 = signal.bode(signal.lti([1, 0], [2, 1]), [0.5, 9.4])
print(f"|H(j0.5)| = {mag2[0]:.2f} dB, phase {ph2[0]:.1f} deg;  |H(j9.4)| = {mag2[1]:.2f} dB  (20 log 0.5 = {20 * np.log10(0.5):.4f})")
t8, y8 = signal.step(signal.lti([1, 0], [2, 1]), T=np.linspace(0, 12, 1201))
print(f"step response y(2) = {np.interp(2, t8, y8):.4f}  (0.5 x 0.368 = 0.184)")
w8 = np.logspace(-2, 1, 300)
_, m8, p8 = signal.bode(signal.lti([1, 0], [2, 1]), w8)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Bode magnitude (dB)", "Step response"))
fig.add_scatter(x=w8, y=np.round(m8, 4), name="|H| dB", row=1, col=1)
fig.add_scatter(x=ds(t8), y=ds(y8), name="y(t) = 0.5 exp(-t/2)", row=1, col=2)
fig.update_xaxes(type="log", title="rad/s", row=1, col=1)
fig.update_xaxes(title="t (s)", row=1, col=2)
show(fig, "High-pass filter s/(2s+1)", None, None)

initial value = 1/2  final value = 0
|H(j0.5)| = -9.03 dB, phase 45.0 deg;  |H(j9.4)| = -6.03 dB  (20 log 0.5 = -6.0206)
step response y(2) = 0.1839  (0.5 x 0.368 = 0.184)


## 8. 低通濾波（`order1Filter.mdl`）

模型：Sine Wave（0.1 rad/s）＋ Sine Wave1（20 rad/s）→ Sum → $\frac{1}{2s+1}$ → Scope，模擬 50 s（ode45、最大步長 0.005）。

$$|H(j0.1)| = \frac{1}{\sqrt{1+0.2^2}} = 0.981,\qquad |H(j20)| = \frac{1}{\sqrt{1+40^2}} = 0.025$$

In [12]:
t9 = np.arange(0, 50.0001, 0.005)
u9 = np.sin(0.1 * t9) + np.sin(20 * t9)
_, y9, _ = signal.lsim(signal.lti([1], [2, 1]), u9, t9)
print(f"|H(j0.1)| = {1 / np.sqrt(1 + 0.2 ** 2):.3f}, |H(j20)| = {1 / np.sqrt(1 + 40 ** 2):.4f}")
print(f"residual ripple of the 20 rad/s wave in y (t > 10 s): {np.std(y9[t9 > 10] - np.interp(t9[t9 > 10], t9, signal.lsim(signal.lti([1], [2, 1]), np.sin(0.1 * t9), t9)[1])) * np.sqrt(2):.4f}")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Input: sin(0.1t) + sin(20t)", "Output of 1/(2s+1)"))
fig.add_scatter(x=t9[::5], y=np.round(u9[::5], 2), name="u", line=dict(width=1), row=1, col=1)
fig.add_scatter(x=ds(t9, 800), y=ds(y9, 800), name="y", row=2, col=1)
fig.update_xaxes(title="t (s)", row=2, col=1)
show(fig, "order1Filter.mdl: low-pass filtering", None, None, h=560)

|H(j0.1)| = 0.981, |H(j20)| = 0.0250


residual ripple of the 20 rad/s wave in y (t > 10 s): 0.0250


## 9. Bode 漸近線、積分器與二階系統

一階 $\frac{1}{Ts+1}$：$\omega \ll 1/T$ 時 0 dB，$\omega \gg 1/T$ 時 $-20\log(\omega T)$（−20 dB/decade）；相位漸近線從 $0.1/T$ 的 0° 直線降到 $10/T$ 的 −90°。
實際曲線在轉角頻率比漸近線低 3 dB。積分器 $\frac{1}{s}$：$-20\log\omega$、相位固定 −90°。

In [13]:
Tn = 1.0
w = np.logspace(-1, 1, 300)
exact = -10 * np.log10(1 + (w * Tn) ** 2)
asym = np.where(w < 1 / Tn, 0, -20 * np.log10(w * Tn))
ph_exact = -np.degrees(np.arctan(w * Tn))
ph_asym = np.clip(-45 * (np.log10(w * Tn) + 1), -90, 0)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_scatter(x=w, y=np.round(exact, 4), name="exact", row=1, col=1)
fig.add_scatter(x=w, y=np.round(asym, 4), name="asymptotic", line=dict(color="black"), row=1, col=1)
fig.add_scatter(x=w, y=np.round(ph_exact, 4), name="exact phase", row=2, col=1)
fig.add_scatter(x=w, y=np.round(ph_asym, 4), name="linear approx.", line=dict(color="black"), row=2, col=1)
fig.update_xaxes(type="log", title="w T", row=2, col=1)
fig.update_yaxes(title="dB", row=1, col=1)
fig.update_yaxes(title="deg", row=2, col=1)
show(fig, "1/(Ts+1): asymptotic vs exact (max error -3 dB at w = 1/T)", None, None, h=560)
print(f"error at corner w = 1/T: {-10 * np.log10(2):.2f} dB;  integrator 1/s at w = 0.1, 1, 10: {[float(round(-20 * np.log10(x), 1)) + 0.0 for x in (0.1, 1, 10)]} dB")

error at corner w = 1/T: -3.01 dB;  integrator 1/s at w = 0.1, 1, 10: [20.0, 0.0, -20.0] dB


二階系統 $G(j\omega) = \dfrac{1}{(1-u^2) + j2\xi u}$，$u = \omega/\omega_n$：阻尼比越小，在 $u \approx 1$ 的共振峰越高。

In [14]:
u = np.logspace(-1, 1, 500)
fig = make_subplots(rows=1, cols=2, subplot_titles=("20 log|G| (dB)", "Phase (deg)"))
for z in (0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0):
    G = 1 / ((1 - u ** 2) + 2j * z * u)
    fig.add_scatter(x=u, y=np.round(20 * np.log10(abs(G)), 3), name=f"zeta = {z}", legendgroup=str(z), row=1, col=1)
    fig.add_scatter(x=u, y=np.round(np.degrees(np.angle(G)), 3), showlegend=False, legendgroup=str(z), row=1, col=2)
fig.update_xaxes(type="log", title="u = w / wn")
show(fig, "Second-order system: frequency response vs damping ratio", None, None, h=460)

網頁上的範例：
$$G(j\omega) = \frac{5\,(1+j0.1\omega)}{j\omega\,(1+j0.5\omega)\left(1 + j0.6\frac{\omega}{50} + \left(\frac{j\omega}{50}\right)^2\right)}$$
由五個因子的漸近線相加（增益 14 dB、原點極點、$\omega=2$ 的極點、$\omega=10$ 的零點、$\omega_n=50,\ \xi=0.3$ 的二階極點）。

In [15]:
w = np.logspace(-1, 2, 600)
jw = 1j * w
G = 5 * (1 + 0.1 * jw) / (jw * (1 + 0.5 * jw) * (1 + 0.6 * jw / 50 + (jw / 50) ** 2))


def asym_db(w):
    a = 20 * np.log10(5) - 20 * np.log10(w)
    a += np.where(w > 2, -20 * np.log10(w / 2), 0)
    a += np.where(w > 10, 20 * np.log10(w / 10), 0)
    a += np.where(w > 50, -40 * np.log10(w / 50), 0)
    return a


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_scatter(x=w, y=np.round(20 * np.log10(abs(G)), 3), name="exact", line=dict(dash="dash"), row=1, col=1)
fig.add_scatter(x=w, y=np.round(asym_db(w), 3), name="asymptotic", line=dict(color="black"), row=1, col=1)
fig.add_scatter(x=w, y=np.round(np.degrees(np.unwrap(np.angle(G))), 3), name="phase", row=2, col=1)
fig.update_xaxes(type="log", title="w (rad/s)", row=2, col=1)
fig.update_yaxes(title="dB", row=1, col=1)
fig.update_yaxes(title="deg", row=2, col=1)
show(fig, "Example: Bode plot built from five factors", None, None, h=560)
print(f"20 log 5 = {20 * np.log10(5):.2f} dB; exact magnitude at w = 1: {20 * np.log10(abs(G[np.argmin(abs(w - 1))])):.2f} dB")

20 log 5 = 13.98 dB; exact magnitude at w = 1: 13.02 dB
